# ¿Qué le recomendamos a Beto?

Un solo supermercado de juguete recorrido por 10 métodos de recomendación. Cada método resuelve una falla del anterior. Todos los números salen de `metodos.py`; los datos, de `generar_datos.py`.

**Respuesta esperada para Beto** (compra Pan y Mantequilla, como Ana y Dani): Leche y Cereal, nunca Pañales.

In [1]:
import json, numpy as np, pandas as pd
import generar_datos, metodos, figuras
pd.set_option('display.precision', 2)

C, B, A = metodos.cargar()
pd.DataFrame(C.astype(int), index=metodos.CLIENTES, columns=metodos.PROD6)  # veces que compró cada producto

,Pan,Mantequilla,Leche,Cereal,Pañales,Toallitas
Ana,5,3,4,2,0,0
Beto,4,2,0,0,0,0
Caro,0,0,3,0,6,4
Dani,2,0,3,3,0,0
Eva,0,0,1,0,2,3


In [2]:
pd.read_csv('datos/productos.csv').set_index('producto')  # atributos; Avena es nueva (nadie la compró)

,Desayuno,Lacteo,Granos,Bebe,Higiene,es_nuevo
producto,,,,,,
Pan,1,0,1,0,0,0
Mantequilla,1,1,0,0,0,0
Leche,1,1,0,1,0,0
Cereal,1,0,1,0,0,0
Pañales,0,0,0,1,1,0
Toallitas,0,0,0,1,1,0
Avena,1,0,1,0,0,1


### El resultado completo en una imagen

In [3]:
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    metodos.main()  # corre los 10 métodos y guarda resultados/resultados.json
res = figuras.cargar()
figuras.resumen_beto(res);

## 1. Apriori

| | |
|---|---|
| **Qué hace** | Cuenta qué productos aparecen juntos en las canastas y arma reglas A → B con soporte, confianza y lift. |
| **Cómo mejora** | Punto de partida: describe la tienda sin ningún modelo. |
| **Su limitación** | No personaliza: a todos los que llevan pan les dice lo mismo. |

In [4]:
pd.DataFrame(metodos.apriori(B))

,regla,soporte,confianza,lift
0,Pañales → Toallitas,0.4,1.00,2.50
1,Toallitas → Pañales,0.4,1.00,2.50
2,Pan → Mantequilla,0.4,0.67,1.67
3,Pan → Cereal,0.4,0.67,1.67
4,Mantequilla → Pan,0.4,1.00,1.67
5,Cereal → Pan,0.4,1.00,1.67
6,Leche → Cereal,0.4,0.50,1.25
7,Leche → Pañales,0.4,0.50,1.25
8,Leche → Toallitas,0.4,0.50,1.25
9,Cereal → Leche,0.4,1.00,1.25


## 2. Vecindario (filtrado colaborativo)

| | |
|---|---|
| **Qué hace** | Compara la canasta de Beto con la de cada cliente (coseno) y promedia lo que compraron sus vecinos. |
| **Cómo mejora** | Personaliza con el historial de cada cliente. |
| **Su limitación** | Similitud 0 entre clientes sin compras en común; costo al cuadrado. |

In [5]:
v = metodos.vecindario(B)
display(pd.Series(v['similitud'], name='coseno con Beto').to_frame().T)
pd.Series(v['puntaje'], name='puntaje Beto').to_frame().T

,Ana,Beto,Caro,Dani,Eva
coseno con Beto,0.71,1.0,0.0,0.41,0.0


,Pan,Mantequilla,Leche,Cereal,Pañales,Toallitas
puntaje Beto,1.0,0.63,1.0,1.0,0.0,0.0


In [6]:
figuras.apriori_vecindario(res);

## 3. SVD

| | |
|---|---|
| **Qué hace** | Descompone la matriz en patrones ocultos y la reconstruye con los k más fuertes. |
| **Cómo mejora** | Conecta clientes de forma indirecta; nacen los embeddings. |
| **Su limitación** | Trata lo no comprado como 'no le gusta'. |

In [7]:
s = metodos.svd(B)
print('valores singulares:', s['valores_singulares'])
pd.DataFrame(s['reconstruida'], index=metodos.CLIENTES, columns=metodos.PROD6)

valores singulares: [2.95, 2.2, 1.12, 0.44, 0.0]


,Pan,Mantequilla,Leche,Cereal,Pañales,Toallitas
Ana,1.22,0.82,0.95,0.92,0.03,0.03
Beto,0.72,0.50,0.30,0.52,-0.21,-0.21
Caro,-0.03,-0.09,1.05,0.08,0.97,0.97
Dani,0.93,0.62,0.84,0.71,0.13,0.13
Eva,-0.03,-0.09,1.05,0.08,0.97,0.97


## 4. SVD de Funk

| | |
|---|---|
| **Qué hace** | Aprende los embeddings con descenso de gradiente solo sobre las celdas conocidas. |
| **Cómo mejora** | Ya no castiga los huecos. |
| **Su limitación** | Con compras solo hay unos: colapsa y predice ≈ 1 en todo. |

In [8]:
f = metodos.funk(B, np.random.default_rng(0))
pd.DataFrame(f['reconstruida'], index=metodos.CLIENTES, columns=metodos.PROD6)

,Pan,Mantequilla,Leche,Cereal,Pañales,Toallitas
Ana,0.98,0.98,0.98,0.98,0.97,0.97
Beto,0.98,0.98,0.98,0.98,0.96,0.96
Caro,0.97,0.96,0.98,0.97,0.98,0.98
Dani,0.98,0.98,0.98,0.98,0.97,0.97
Eva,0.97,0.96,0.98,0.97,0.98,0.98


## 5. ALS

| | |
|---|---|
| **Qué hace** | Mismo objetivo que Funk, resuelto por turnos con regresión exacta. |
| **Cómo mejora** | Rápido y paralelizable. |
| **Su limitación** | Mismo colapso: cambia el cómo, no el qué. |

In [9]:
a = metodos.als(B, np.random.default_rng(1))
pd.DataFrame(a['reconstruida'], index=metodos.CLIENTES, columns=metodos.PROD6)

,Pan,Mantequilla,Leche,Cereal,Pañales,Toallitas
Ana,0.98,0.97,0.98,0.96,0.95,0.95
Beto,0.95,0.94,0.95,0.93,0.93,0.93
Caro,0.98,0.96,0.98,0.96,0.96,0.96
Dani,0.97,0.96,0.97,0.95,0.94,0.94
Eva,0.98,0.96,0.98,0.96,0.96,0.96


## 6. ALS implicit

| | |
|---|---|
| **Qué hace** | Usa todas las celdas con una confianza c = 1 + α·veces: los huecos son un 'no' débil. |
| **Cómo mejora** | Primer método que acierta y ordena para Beto. |
| **Su limitación** | No conoce productos ni clientes nuevos; ignora atributos y orden. |

In [10]:
rng = np.random.default_rng(0); metodos.funk(B, rng)  # mismo orden de semillas que metodos.main
ai = metodos.als_implicit(C, B, rng)
print('pérdida por vuelta:', ai['perdida'])
pd.DataFrame(ai['reconstruida'], index=metodos.CLIENTES, columns=metodos.PROD6)

pérdida por vuelta: {1: 60.6, 2: 41.8, 3: 33.3, 5: 24.3, 15: 12.3}


,Pan,Mantequilla,Leche,Cereal,Pañales,Toallitas
Ana,1.03,0.99,0.99,1.01,0.16,0.16
Beto,0.98,0.97,0.31,0.74,-0.41,-0.41
Caro,-0.18,-0.21,1.01,0.23,1.00,1.00
Dani,0.97,0.93,1.00,0.97,0.20,0.21
Eva,-0.18,-0.21,0.99,0.23,0.99,0.99


In [11]:
figuras.factorizaciones(res);

## 7. Basado en contenido

| | |
|---|---|
| **Qué hace** | Perfil del cliente = promedio de atributos de lo que compró; recomienda los productos más parecidos. |
| **Cómo mejora** | Recomienda la Avena sin una sola venta. |
| **Su limitación** | Solo 'más de lo mismo'; no aprende del comportamiento de otros. |

In [12]:
c = metodos.contenido(B, A)
pd.DataFrame({'Beto': c['beto'], 'Fede (nuevo)': c['fede']}).T

,Pan,Mantequilla,Leche,Cereal,Pañales,Toallitas,Avena
Beto,0.87,0.87,0.71,0.87,0.0,0.0,0.87
Fede (nuevo),0.00,0.00,0.41,0.00,1.0,1.0,0.00


## 8. Two-Tower

| | |
|---|---|
| **Qué hace** | Dos redes (cliente y producto) producen embeddings; P(compra) = σ(u·v). Entrenada con 200 clientes simulados. |
| **Cómo mejora** | Une comportamiento y atributos: resuelve producto y cliente nuevos. |
| **Su limitación** | No ve el orden ni el momento de la compra. |

In [13]:
t = metodos.two_tower(B, A)
print('pérdida:', t['perdida'])
pd.DataFrame({'Beto': t['beto'], 'Fede (nuevo)': t['fede']}).T

pérdida: {1: 0.711, 100: 0.421, 500: 0.413, 2000: 0.401}


,Pan,Mantequilla,Leche,Cereal,Pañales,Toallitas,Avena
Beto,0.93,0.99,0.60,0.69,0.02,0.03,0.84
Fede (nuevo),0.09,0.11,0.63,0.19,0.91,0.89,0.16


In [14]:
figuras.cold_start(res);

## 9. GRU4Rec

| | |
|---|---|
| **Qué hace** | Lee el carrito en orden con una memoria (GRU) y predice el siguiente producto. 600 carritos simulados. |
| **Cómo mejora** | Responde '¿qué agrega ahora?'. |
| **Su limitación** | Memoria que se diluye en carritos largos; entrenamiento secuencial. |

In [15]:
g = metodos.gru4rec()
pd.DataFrame(g['siguiente']).T

,Pan,Mantequilla,Leche,Cereal,Pañales,Toallitas
Pan → Cereal,0.04,0.02,0.79,0.13,0.01,0.02
Pan → Leche,0.06,0.01,0.04,0.86,0.02,0.00
Pañales → Leche,0.02,0.01,0.06,0.01,0.02,0.88


## 10. SASRec

| | |
|---|---|
| **Qué hace** | Atención: en cada paso mira directamente a todos los productos anteriores y los pondera. |
| **Cómo mejora** | Conecta pasos lejanos, entrena en paralelo, pesos inspeccionables. |
| **Su limitación** | Mucho dato; la versión base solo usa IDs. |

In [16]:
sa = metodos.sasrec()
display(pd.DataFrame(sa['siguiente']).T)
pd.DataFrame(sa['atencion']).T

,Pan,Mantequilla,Leche,Cereal,Pañales,Toallitas
Pan → Cereal,0.04,0.02,0.77,0.14,0.01,0.02
Pan → Leche,0.06,0.02,0.04,0.86,0.02,0.00
Pañales → Leche,0.02,0.01,0.06,0.01,0.02,0.88


,Pan,Cereal,Leche,Pañales
Pan → Cereal,0.55,0.45,NaN,NaN
Pan → Leche,0.67,NaN,0.33,NaN
Pañales → Leche,NaN,NaN,0.56,0.44


In [17]:
figuras.secuencias(res);

## Conclusión

Los métodos 1 a 6 responden *¿qué le gusta a Beto?*; el 7 y el 8 agregan *¿y lo nuevo?*; el 9 y el 10 responden *¿qué hará ahora?*. En producción se combinan: uno genera candidatos (ALS o Two-Tower) y otro los ordena según la sesión (GRU4Rec o SASRec).